# Imports

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import copy
import json
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import thop
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torchvision.models import MobileNet_V3_Large_Weights
from torchvision.models.segmentation import lraspp_mobilenet_v3_large

from uavloc.features.segmentation import CLASSES, ENCODER_STRIDE_PX, class_probabilities, load_unet
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.track import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

# Params

In [ ]:
TAG = "lraspp"
REFERENCE_CKPTS = {"satellite": "model-round3-bn-satellite.pt", "drone": "model-round3-bn-drone.pt"}
HOLDOUT_VILLAGE = "bushcha-50p2422-25p8305"
STRIP_VILLAGE = "birky-49p9105-23p9192"
STRIP_ROWS = 640
GAP_PX = 64
GOOGLE_DATE = "google"
DRONE_VAL_BLOCKS = [0, 4]
DRONE_P = 0.5
DRONE_GSD_M = 0.10
SAT_CROP_PX, SAT_BATCH = 512, 4
DRONE_CROP_PX, DRONE_BATCH = 256, 8
STEPS = 3000
EVAL_EVERY = 250
LR = 5e-4
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
RARE_CROP_P = 0.3
BN_BATCHES = 60
BN_SEED = 123
SEED = 0
PROB_FLOOR = 1e-8
BENCH_DEVICES = ["mps", "cpu"]
BENCH_WARMUP, BENCH_RUNS = 5, 30
BENCH_SAT_BATCH = 8
SHOWCASE_FRAMES = 4
SERIES_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

# Data

In [ ]:
DERIVED = REPO_ROOT / "data" / "derived"
VILLAGES = DERIVED / "ua-village-annotated"
DRONE = DERIVED / "ua-drone-annotated"
BASELINE = REPO_ROOT / "pocs" / "village-seg-baseline"
HERE = REPO_ROOT / "pocs" / "mobilenet-lraspp-speed"
FIGS = HERE / "figures"
FIGS.mkdir(exist_ok=True)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(SEED)
DEVICE

In [ ]:
sat = vs.load_satellite(VILLAGES, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_VILLAGE, STRIP_VILLAGE, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
sat_sampler = vs.SatelliteSampler(sat_train, SAT_CROP_PX, RARE_CROP_P)
[(s.loc.split("-")[0], s.date) for s in sat_train]

In [ ]:
drone = vs.load_drone(DRONE, DERIVED)
vs.split_by_footprint(drone, DRONE_VAL_BLOCKS)
round3_splits = json.loads((BASELINE / "results-round3.json").read_text())["splits"]
assert [(s.loc, s.date) for s in sat_train] == [tuple(p) for p in round3_splits["sat_train"]]
assert [d.name for d in drone if d.split == "train"] == round3_splits["drone_train"]
assert [d.name for d in drone if d.split == "val"] == round3_splits["drone_val"]
pd.Series([d.split for d in drone]).value_counts()

In [ ]:
drone_train = [d for d in drone if d.split == "train"]
drone_val = [d for d in drone if d.split == "val"]
drone_sampler = vs.DroneSampler(drone_train, DRONE_GSD_M, DRONE_CROP_PX, RARE_CROP_P)

In [ ]:
def unbanned_label(s: vs.SatImage) -> np.ndarray:
    """Label with the held-out strip and its gap ignored, so train scores cover only trained-on ground."""
    lab = s.lab.copy()
    lab[s.banned] = vs.IGNORE
    return lab

In [ ]:
strip = [s for s in sat if s.loc == STRIP_VILLAGE]
EVAL = {
    "sat-train": [(s.img, unbanned_label(s)) for s in sat_train],
    "drone-train": [vs.drone_at_gsd(d, DRONE_GSD_M) for d in drone_train],
    "birky-strip": [(s.img[:STRIP_ROWS], s.lab[:STRIP_ROWS]) for s in strip if s.date != GOOGLE_DATE],
    "birky-strip-google": [(s.img[:STRIP_ROWS], s.lab[:STRIP_ROWS]) for s in strip if s.date == GOOGLE_DATE],
    "bushcha": [(s.img, s.lab) for s in sat if s.loc == HOLDOUT_VILLAGE],
    "drone-val": [vs.drone_at_gsd(d, DRONE_GSD_M) for d in drone_val],
}
SET_SPLIT = {n: "train" if n.endswith("-train") else "val" for n in EVAL}
SET_DOMAIN = {n: "drone" if n.startswith("drone") else "satellite" for n in EVAL}
pd.DataFrame({"images": {n: len(v) for n, v in EVAL.items()}, "split": SET_SPLIT, "domain": SET_DOMAIN})

# Model

In [ ]:
class LRASPPLogits(nn.Module):
    """torchvision's LR-ASPP returns a dict; the `uavloc` helpers expect logits `(B, C, H, W)`."""

    def __init__(self, n_classes: int, pretrained_backbone: bool) -> None:
        super().__init__()
        weights = MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained_backbone else None
        self.net = lraspp_mobilenet_v3_large(weights=None, weights_backbone=weights, num_classes=n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)["out"]

In [ ]:
def dice_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    ok = y != vs.IGNORE
    p = logits.softmax(1) * ok[:, None]
    onehot = F.one_hot(y.clamp(max=len(CLASSES) - 1), len(CLASSES)).permute(0, 3, 1, 2) * ok[:, None]
    present = onehot.sum((0, 2, 3)) > 0
    inter, den = (p * onehot).sum((0, 2, 3)), p.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    return (1 - (2 * inter + 1) / (den + 1))[present].mean()

In [ ]:
def seg_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """The round-3 training loss: CE (ignore 255) + soft Dice."""
    return F.cross_entropy(logits, y, ignore_index=vs.IGNORE) + dice_loss(logits, y)

In [ ]:
def image_loss(probs: np.ndarray, lab: np.ndarray) -> float:
    """`seg_loss` of a whole-image prediction; log-probabilities stand in for logits, softmax is unchanged."""
    logits = torch.from_numpy(probs).clamp_min(PROB_FLOOR).log()[None]
    return seg_loss(logits, torch.from_numpy(lab.astype(np.int64))[None]).item()

In [ ]:
def bn_pair(model: nn.Module) -> dict[str, nn.Module]:
    """Same weights, BatchNorm statistics re-estimated per domain from its own training crops."""
    return {"satellite": vs.domain_bn(model, sat_sampler, SAT_BATCH, BN_BATCHES, BN_SEED, DEVICE),
            "drone": vs.domain_bn(model, drone_sampler, DRONE_BATCH, BN_BATCHES, BN_SEED, DEVICE)}

In [ ]:
def score_set(model: nn.Module, items: list[tuple[np.ndarray, np.ndarray]]) -> tuple[dict, list[np.ndarray]]:
    """Per-class IoU, mIoU and mean whole-image loss over one eval set, plus the predictions."""
    preds, losses = [], []
    for img, lab in items:
        probs = class_probabilities(model, img, DEVICE)
        preds.append(probs.argmax(0).astype(np.uint8))
        losses.append(image_loss(probs, lab))
    return {**vs.iou_scores(preds, [lab for _, lab in items]), "loss": float(np.mean(losses))}, preds

In [ ]:
def evaluate(models: dict[str, nn.Module]) -> tuple[dict, dict]:
    """Satellite sets with the satellite-BN model, drone sets with the drone-BN model."""
    scores, preds = {}, {}
    for name, items in EVAL.items():
        scores[name], preds[name] = score_set(models[SET_DOMAIN[name]], items)
    return scores, preds

In [ ]:
def metric_line(ev: dict) -> str:
    return "  ".join(f"{n} {ev[n]['loss']:.3f}/{ev[n]['miou']:.3f}" for n in EVAL)

In [ ]:
reference_models = {d: load_unet(BASELINE / c, DEVICE) for d, c in REFERENCE_CKPTS.items()}
t0 = time.time()
reference, reference_preds = evaluate(reference_models)
reference_eval_minutes = (time.time() - t0) / 60
metric_line(reference), f"{reference_eval_minutes:.1f} min"

In [ ]:
model = LRASPPLogits(len(CLASSES), pretrained_backbone=True).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=WARMUP_FRAC)

# Run

In [ ]:
rng = np.random.default_rng(SEED)
history, batch_losses, t0 = [], {"satellite": [], "drone": []}, time.time()
for step in range(1, STEPS + 1):
    model.train()
    domain = "drone" if rng.random() < DRONE_P else "satellite"
    x, y = (vs.batch_of(drone_sampler, rng, DRONE_BATCH) if domain == "drone"
            else vs.batch_of(sat_sampler, rng, SAT_BATCH))
    loss = seg_loss(model(x.to(DEVICE)), y.to(DEVICE))
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    batch_losses[domain].append(loss.item())
    if step % EVAL_EVERY == 0 or step == STEPS:
        minutes = (time.time() - t0) / 60
        ev, _ = evaluate(bn_pair(model))
        history.append({"step": step, "minutes": minutes,
                        "batch_loss": {k: float(np.mean(v)) for k, v in batch_losses.items() if v}, "eval": ev})
        print(f"step {step:4d}  {metric_line(ev)}  [{minutes:.0f} min]", flush=True)
        batch_losses = {"satellite": [], "drone": []}
train_minutes = (time.time() - t0) / 60

In [ ]:
final_models = bn_pair(model)
torch.save(model.state_dict(), HERE / f"model-{TAG}.pt")
for domain, m in final_models.items():
    torch.save(m.state_dict(), HERE / f"model-{TAG}-bn-{domain}.pt")
final, final_preds = evaluate(final_models)
metric_line(final)

In [ ]:
def history_frame(history: list[dict]) -> pd.DataFrame:
    """One row per (step, eval set) plus one per (step, domain) for the running training-batch loss."""
    rows = []
    for h in history:
        for name, ev in h["eval"].items():
            rows.append({"step": h["step"], "minutes": h["minutes"], "set": name, "split": SET_SPLIT[name],
                         "domain": SET_DOMAIN[name], "loss": ev["loss"], "miou": ev["miou"],
                         **{f"iou_{c}": v for c, v in ev["iou"].items()}})
        for domain, loss in h["batch_loss"].items():
            rows.append({"step": h["step"], "minutes": h["minutes"], "set": f"{domain}-batch",
                         "split": "train-batch", "domain": domain, "loss": loss})
    return pd.DataFrame(rows)

In [ ]:
hist = history_frame(history)
hist.to_csv(HERE / f"history-{TAG}.csv", index=False)
hist.pivot(index="step", columns="set", values="miou").dropna(axis=1).round(3)

In [ ]:
def sync(device: torch.device) -> None:
    if device.type == "mps":
        torch.mps.synchronize()

In [ ]:
@torch.no_grad()
def latency_ms(model: nn.Module, x: torch.Tensor, device: torch.device) -> float:
    """Median forward-pass wall time over `BENCH_RUNS`, after `BENCH_WARMUP` untimed passes."""
    model, x = copy.deepcopy(model).to(device).eval(), x.to(device)
    for _ in range(BENCH_WARMUP):
        model(x)
    sync(device)
    times = []
    for _ in range(BENCH_RUNS):
        t = time.perf_counter()
        model(x)
        sync(device)
        times.append(time.perf_counter() - t)
    return float(np.median(times)) * 1000

In [ ]:
def padded(px: int) -> int:
    return -(-px // ENCODER_STRIDE_PX) * ENCODER_STRIDE_PX

In [ ]:
frame_h, frame_w = EVAL["drone-val"][0][0].shape[:2]
DRONE_INPUT = torch.randn(1, 3, padded(frame_h), padded(frame_w))
SAT_INPUT = torch.randn(BENCH_SAT_BATCH, 3, SAT_CROP_PX, SAT_CROP_PX)
BENCH_MODELS = {"U-Net/ResNet34 (round 3)": reference_models["drone"], "MobileNetV3-L + LR-ASPP": final_models["drone"]}
tuple(DRONE_INPUT.shape), tuple(SAT_INPUT.shape)

In [ ]:
def speed_row(name: str, model: nn.Module, device_name: str) -> dict:
    device = torch.device(device_name)
    macs, params = thop.profile(copy.deepcopy(model).cpu(), (DRONE_INPUT,), verbose=False)
    return {"model": name, "device": device_name, "params_m": params / 1e6, "gmacs_drone_frame": macs / 1e9,
            "drone_frame_ms": latency_ms(model, DRONE_INPUT, device),
            "sat_batch_ms": latency_ms(model, SAT_INPUT, device)}

In [ ]:
speed = pd.DataFrame([speed_row(n, m, d) for n, m in BENCH_MODELS.items() for d in BENCH_DEVICES])
unet_ms = speed[speed["model"].str.startswith("U-Net")].set_index("device")
speed["drone_frame_speedup"] = unet_ms.loc[speed["device"], "drone_frame_ms"].values / speed["drone_frame_ms"]
speed["sat_batch_speedup"] = unet_ms.loc[speed["device"], "sat_batch_ms"].values / speed["sat_batch_ms"]
speed.round(2)

In [ ]:
results = {
    "tag": TAG, "reference": REFERENCE_CKPTS, "train_minutes": train_minutes,
    "reference_eval_minutes": reference_eval_minutes,
    "params": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict))
               and k not in {"CLASSES", "EVAL", "SET_SPLIT", "SET_DOMAIN", "BENCH_MODELS"}},
    "splits": {"sat_train": [(s.loc, s.date) for s in sat_train], "drone_train": [d.name for d in drone_train],
               "drone_val": [d.name for d in drone_val]},
    "reference_scores": reference, "final_scores": final, "history": history, "speed": speed.to_dict("records"),
}
(HERE / f"results-{TAG}.json").write_text(json.dumps(results, indent=1, default=str))

# Plots

In [ ]:
def colour(lab: np.ndarray) -> np.ndarray:
    out = np.full((*lab.shape, 3), 255, np.uint8)
    ok = lab != vs.IGNORE
    out[ok] = SEG_PALETTE[lab[ok]]
    return out

In [ ]:
def plot_metric(ax: plt.Axes, domain: str, metric: str) -> None:
    """One line per eval set of `domain` (solid: LR-ASPP over steps; dashed: final round-3 U-Net)."""
    sets = [n for n in EVAL if SET_DOMAIN[n] == domain]
    for name, c in zip(sets, SERIES_COLOURS):
        rows = hist[hist["set"] == name]
        ax.plot(rows["step"], rows[metric], "-o", color=c, lw=2, ms=4, label=f"{name} ({SET_SPLIT[name]})")
        ax.axhline(reference[name][metric], color=c, ls="--", lw=1.2)
    if metric == "loss":
        rows = hist[hist["set"] == f"{domain}-batch"]
        ax.plot(rows["step"], rows["loss"], ":", color="0.45", lw=2, label="training batches (augmented)")
    ax.set_title(f"{domain}: {'loss (CE + Dice)' if metric == 'loss' else 'mIoU'}")
    ax.set_xlabel("step"); ax.grid(alpha=.25); ax.legend(fontsize=8)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for col, domain in enumerate(["satellite", "drone"]):
    plot_metric(axes[0, col], domain, "loss")
    plot_metric(axes[1, col], domain, "miou")
    axes[1, col].set_ylim(0, 1)
fig.suptitle("MobileNetV3-L + LR-ASPP, per-domain BN   (dashed = round-3 U-Net, final)")
plt.tight_layout(); plt.savefig(FIGS / f"{TAG}-curves.png", dpi=110); plt.show()

In [ ]:
rows = []
for name in EVAL:
    for c, r in reference[name]["iou"].items():
        rows.append({"set": name, "metric": c, "U-Net r3": r, "LR-ASPP": final[name]["iou"][c]})
    for m in ("miou", "loss"):
        rows.append({"set": name, "metric": m, "U-Net r3": reference[name][m], "LR-ASPP": final[name][m]})
table = pd.DataFrame(rows).set_index(["set", "metric"])
table.assign(delta=table["LR-ASPP"] - table["U-Net r3"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, col, title in zip(axes, ["drone_frame_ms", "sat_batch_ms"],
                          [f"drone frame {tuple(DRONE_INPUT.shape[2:])}, batch 1", f"satellite {BENCH_SAT_BATCH} x {SAT_CROP_PX} px"]):
    pivot = speed.pivot(index="device", columns="model", values=col)[list(BENCH_MODELS)]
    pivot.plot.barh(ax=ax, color=SERIES_COLOURS[:2], width=.7)
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.0f ms", padding=3, fontsize=8)
    ax.set_title(title); ax.set_xlabel("median forward pass, ms"); ax.grid(axis="x", alpha=.25)
    ax.margins(x=.2); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGS / f"{TAG}-speed.png", dpi=110); plt.show()

In [ ]:
legend = [plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))]
picks = np.linspace(0, len(drone_val) - 1, SHOWCASE_FRAMES).astype(int)
fig, axes = plt.subplots(SHOWCASE_FRAMES, 4, figsize=(18, 2.7 * SHOWCASE_FRAMES), squeeze=False)
for r, k in enumerate(picks):
    img, lab = EVAL["drone-val"][k]
    panels = [(img, drone_val[k].name), (colour(lab), "label"),
              (colour(reference_preds["drone-val"][k]), "U-Net r3"), (colour(final_preds["drone-val"][k]), "LR-ASPP")]
    for ax, (p, t) in zip(axes[r], panels):
        ax.imshow(p); ax.set_title(t, fontsize=9); ax.axis("off")
fig.legend(legend, CLASSES, loc="lower center", ncol=len(CLASSES))
plt.tight_layout(rect=(0, .03, 1, 1)); plt.savefig(FIGS / f"{TAG}-drone-val.png", dpi=80, bbox_inches="tight"); plt.show()

In [ ]:
(img, lab), = EVAL["birky-strip-google"]
panels = [(img, "Google (held-out strip)"), (colour(lab), "label"),
          (colour(reference_preds["birky-strip-google"][0]), "U-Net r3"),
          (colour(final_preds["birky-strip-google"][0]), "LR-ASPP")]
fig, axes = plt.subplots(len(panels), 1, figsize=(18, 3.8 * len(panels)))
for ax, (p, t) in zip(axes, panels):
    ax.imshow(p); ax.set_title(t); ax.axis("off")
fig.legend(legend, CLASSES, loc="lower center", ncol=len(CLASSES))
plt.tight_layout(rect=(0, .02, 1, 1)); plt.savefig(FIGS / f"{TAG}-birky-strip-google.png", dpi=80, bbox_inches="tight"); plt.show()